In [1]:

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc


# -------------------------------------------------
# Device selection (ROCm or CPU)
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
! pip install kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

In [5]:
import pandas as pd

# Path to the dataset file
file_path = "/root/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3/creditcard.csv"

# Load into a DataFrame
df = pd.read_csv(file_path)

# Take a quick look
print(df.head())
print(df.info())


   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [ ]:
# -------------------------------------------------
# Load dataset
# -------------------------------------------------

# The original dataset can be found at https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
#df = pd.read_csv('creditcard.csv')
#df.head()

In [7]:
val = df[df["Class"] == 1]
ones = val.shape[0]
val = df[df["Class"] == 0]
zeros = val.shape[0]

print(f"Percent frauds: {(ones)/(zeros+ones)*100}%")

Percent frauds: 0.1727485630620034%


In [8]:
# -------------------------------------------------
# Prepare targets
# -------------------------------------------------

dat = df.drop(columns=["Class","Time"])
target = df["Class"].values
target

array([0, 0, 0, ..., 0, 0, 0], shape=(284807,))

In [9]:
# -------------------------------------------------
# Prepare features
# -------------------------------------------------

max = dat["Amount"].max()
min = dat["Amount"].min()
dat["Amount"] = (dat["Amount"] - min) / (max - min)
dat.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0.005824
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,0.000105
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,0.014739
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0.004807
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0.002724


In [10]:
# -------------------------------------------------
# Final feature and target arrays
# -------------------------------------------------

X = dat.values
y = target
X,y

(array([[-1.35980713e+00, -7.27811733e-02,  2.53634674e+00, ...,
          1.33558377e-01, -2.10530535e-02,  5.82379309e-03],
        [ 1.19185711e+00,  2.66150712e-01,  1.66480113e-01, ...,
         -8.98309914e-03,  1.47241692e-02,  1.04705276e-04],
        [-1.35835406e+00, -1.34016307e+00,  1.77320934e+00, ...,
         -5.53527940e-02, -5.97518406e-02,  1.47389219e-02],
        ...,
        [ 1.91956501e+00, -3.01253846e-01, -3.24963981e+00, ...,
          4.45477214e-03, -2.65608286e-02,  2.64215395e-03],
        [-2.40440050e-01,  5.30482513e-01,  7.02510230e-01, ...,
          1.08820735e-01,  1.04532821e-01,  3.89238944e-04],
        [-5.33412522e-01, -1.89733337e-01,  7.03337367e-01, ...,
         -2.41530880e-03,  1.36489143e-02,  8.44648509e-03]],
       shape=(284807, 29)),
 array([0, 0, 0, ..., 0, 0, 0], shape=(284807,)))

In [11]:
# -------------------------------------------------
# Sanity check
# -------------------------------------------------

print(dat[["V1", "V21", "Amount"]].describe())
print(dat.isna().sum())

                 V1           V21         Amount
count  2.848070e+05  2.848070e+05  284807.000000
mean   1.175161e-15  1.628620e-16       0.003439
std    1.958696e+00  7.345240e-01       0.009736
min   -5.640751e+01 -3.483038e+01       0.000000
25%   -9.203734e-01 -2.283949e-01       0.000218
50%    1.810880e-02 -2.945017e-02       0.000856
75%    1.315642e+00  1.863772e-01       0.003004
max    2.454930e+00  2.720284e+01       1.000000
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
dtype: int64


In [12]:
# -------------------------------------------------
# Train-test split
# -------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
# -------------------------------------------------
# Convert to PyTorch tensors and move to device
# -------------------------------------------------
X_train_gpu = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_gpu  = torch.tensor(X_test,  dtype=torch.float32).to(device)

y_train_gpu = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_gpu  = torch.tensor(y_test,  dtype=torch.long).to(device)
y_train_gpu = y_train_gpu.float().unsqueeze(1)
y_test_gpu  = y_test_gpu.float().unsqueeze(1)


In [14]:
#-------------------------------------------------
# Hyperparameters
#-------------------------------------------------
epochs = 30
batch_size = 64
lr = 0.001
input_size = X_train_gpu.shape[1]

# -------------------------------------------------
# Define a simple MLP model
# -------------------------------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)

# -------------------------------------------------
# Loss and optimizer
# -------------------------------------------------

num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
pos_weight = torch.tensor(num_neg / num_pos).to(device)  # ~578 for this dataset

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight) 

# Alternative without pos_weight. The result is much worse
#criterion = nn.BCEWithLogitsLoss()


optimizer = optim.Adam(model.parameters(), lr=lr)

# -------------------------------------------------
# Training loop
# -------------------------------------------------


dataset = torch.utils.data.TensorDataset(X_train_gpu, y_train_gpu)
loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(loader):.4f}")

#


Epoch 1/30 - Loss: 0.4190
Epoch 2/30 - Loss: 0.2631
Epoch 3/30 - Loss: 0.2340
Epoch 4/30 - Loss: 0.2140
Epoch 5/30 - Loss: 0.1701
Epoch 6/30 - Loss: 0.1666
Epoch 7/30 - Loss: 0.1359
Epoch 8/30 - Loss: 0.1079
Epoch 9/30 - Loss: 0.1212
Epoch 10/30 - Loss: 0.1296
Epoch 11/30 - Loss: 0.1121
Epoch 12/30 - Loss: 0.0935
Epoch 13/30 - Loss: 0.0882
Epoch 14/30 - Loss: 0.0864
Epoch 15/30 - Loss: 0.1128
Epoch 16/30 - Loss: 0.0678
Epoch 17/30 - Loss: 0.0865
Epoch 18/30 - Loss: 0.0768
Epoch 19/30 - Loss: 0.0903
Epoch 20/30 - Loss: 0.0835
Epoch 21/30 - Loss: 0.0801
Epoch 22/30 - Loss: 0.0661
Epoch 23/30 - Loss: 0.0778
Epoch 24/30 - Loss: 0.0511
Epoch 25/30 - Loss: 0.0640
Epoch 26/30 - Loss: 0.0664
Epoch 27/30 - Loss: 0.0763
Epoch 28/30 - Loss: 0.0573
Epoch 29/30 - Loss: 0.0676
Epoch 30/30 - Loss: 0.0511


In [15]:
# Release GPU memory
del X_train_gpu
del y_train_gpu
X_test_gpu  = torch.tensor(X_test,  dtype=torch.float32).to(device)

y_test_gpu  = torch.tensor(y_test,  dtype=torch.long).to(device)
y_test_gpu  = y_test_gpu.float().unsqueeze(1)

# Evaluation

In [16]:

model.eval()
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for i in range(0, len(X_test_gpu), batch_size):
        xb = X_test_gpu[i:i+batch_size]
        yb = y_test_gpu[i:i+batch_size]

        logits = model(xb)                  # raw logits
        probs = torch.sigmoid(logits)       # probabilities [0-1]
        preds = (probs > 0.5).float()       # binary predictions (threshold 0.5)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

all_probs = np.array(all_probs).squeeze()
all_preds = np.array(all_preds).squeeze()
all_labels = np.array(all_labels).squeeze()

# Basic report
print(classification_report(all_labels, all_preds, target_names=['Non-Fraud', 'Fraud']))

# AUC scores (very important for imbalance)
print("ROC AUC:", roc_auc_score(all_labels, all_probs))
precision, recall, _ = precision_recall_curve(all_labels, all_probs)
print("PR AUC:", auc(recall, precision))

              precision    recall  f1-score   support

   Non-Fraud       1.00      0.99      1.00     56864
       Fraud       0.16      0.89      0.27        98

    accuracy                           0.99     56962
   macro avg       0.58      0.94      0.63     56962
weighted avg       1.00      0.99      0.99     56962

ROC AUC: 0.9811213543520955
PR AUC: 0.7664849677075728
